In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
import json
import time

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini")

In [2]:
plain_prompt = """다음 제품 리뷰를 분석해줘. 전체 감정(긍정/부정/혼합), 1~5점 점수, 장점목록, 단점목록, 핵심 키워드 3개를 알려줘.
리뷰 : '이 노트북 정말 가벼워서 좋아요! 다만 키보드 타건감이 아쉽네요'
유효한 json만 출력하세요
"""

llm.invoke([HumanMessage(content=plain_prompt)])

AIMessage(content='```json\n{\n  "전체 감정": "혼합",\n  "점수": 4,\n  "장점목록": ["가벼움"],\n  "단점목록": ["키보드 타건감 아쉬움"],\n  "핵심 키워드": ["가벼움", "노트북", "키보드"]\n}\n```', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 98, 'total_tokens': 178, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ca3e7d71bf', 'id': 'chatcmpl-DNGMiupCapwEtSJuLUafBhT0K0wsk', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d24ad-304e-7c83-bcc7-ae4de21a9d0d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 98, 'output_tokens': 80, 'total_tokens': 178, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

In [6]:
markdown_prompt = """
# 제품 리뷰 감정 분석
## 입력리뷰
> 이 노트북 정말 가벼워서 좋아요! 다만 키보드 타건감이 아쉽네요

## 분석항목
- **overall_sentiment** : 긍정/부정/혼합 중 하나
- **score** : 1~5점수
- **pros** : 장점 리스트
- **cons** : 단점 리스트
- **keywords** : 핵심 키워드 3개

## 출력 형식
유효한 json만 출력하세요. 다른 텍스트를 포함하지 마세요
"""

print(llm.invoke([HumanMessage(content=markdown_prompt)]).content)

```json
{
  "overall_sentiment": "혼합",
  "score": 4,
  "pros": ["가벼움"],
  "cons": ["키보드 타건감"],
  "keywords": ["노트북", "가벼움", "키보드"]
}
```


In [5]:
json_prompt = """
다음 제품 리뷰를 분석해서 json 형식으로 출력하세요

리뷰 : "이 노트북 정말 가벼워서 좋아요! 다만 키보드 타건감이 아쉽네요"

출력 형식
{
    "overall_sentiment" : 긍정/부정/혼합 중 하나,
    "score" : 1~5점수,
    "pros" : [장점],
    "cons" : [단점],
    "keywords" : [키워드1, 키워드2, 키워드3]
}
"""

print(llm.invoke([HumanMessage(content=json_prompt)]).content)

```json
{
    "overall_sentiment": "혼합",
    "score": 4,
    "pros": ["가벼움"],
    "cons": ["키보드 타건감"],
    "keywords": ["노트북", "가벼움", "키보드"]
}
```


In [8]:
# 회의록 -> 구조화된 데이터로 변환
json_structure = {
    "date": "YYYY-MM-DD",
    "attendees": ["이름1", "이름2"],
    "agenda": ["논의 주제"],
    "decisions": ["확정된 사항"],
    "action_items": [
        {
            "assignee": "담당자",
            "task": "해야 할 일",
            "deadline": "YYYY-MM-DD"
        }
    ]
}

In [9]:
def generate_meeting_minute(raw_text: str):
    messages = [
        SystemMessage(content="당신은 회의록을 구조화하는 AI입니다. 반드시 JSON으로만 응답하세요."),
        HumanMessage(content=f"""
            다음 회의 내용을 JSON으로 변환하세요.
            
            규칙:
            - 날짜는 YYYY-MM-DD 형식으로 변환 (연도는 2024로 가정)
            - 참석자는 이름만 추출
            - agenda: 논의된 주제
            - decisions: 확정된 사항
            - action_items: 담당자, 작업, 기한 포함
            
            출력 형식:
            {json_structure}
            
            회의 내용:
            {raw_text}
            """)
    ]

    response = llm.invoke(messages)

    # JSON 파싱까지 포함
    try:
        return json.loads(response.content)
    except:
        # 실패 시 raw 반환 (디버깅용)
        return {
            "error": "JSON 파싱 실패",
            "raw_response": response.content
        }

In [10]:
text = """
3월 15일 마케팅팀 주간 회의. 참석: 김팀장, 이대리, 박사원.
신규 SNS 캠페인 예산 5000만원 확정.
이대리가 3월 22일까지 시안 준비.
박사원은 경쟁사 분석 보고서 3월 20일까지.
"""

result = generate_meeting_minute(text)

print(result)

{'date': '2024-03-15', 'attendees': ['김팀장', '이대리', '박사원'], 'agenda': ['신규 SNS 캠페인 예산'], 'decisions': ['신규 SNS 캠페인 예산 5000만원 확정'], 'action_items': [{'assignee': '이대리', 'task': '시안 준비', 'deadline': '2024-03-22'}, {'assignee': '박사원', 'task': '경쟁사 분석 보고서', 'deadline': '2024-03-20'}]}


In [15]:
# 제약에 대한 조건을 추가하는 클래스
class ConstrainedPrompt:
    def __init__(self, base_instruction):
        self.instruction = base_instruction
        self.constraints = []

    def add_length(self, description):
        self.constraints.append(f"[길이] {description}")
        return self

    def add_content(self, description):
        self.constraints.append(f"[내용] {description}")
        return self

    def add_format(self, description):
        self.constraints.append(f"[형식] {description}")
        return self

    def add_style(self, description):
        self.constraints.append(f"[스타일] {description}")
        return self

    def build(self):
        parts = [self.instruction, "\n제약조건:"]
        for c in self.constraints:
            parts.append(f" - {c}")
        return "\n".join(parts)

    # dict 형식으로 받겠다
    def execute(self, **kwargs):
        prompt = self.build()
        return llm.invoke([HumanMessage(content=prompt)]).content

In [16]:
result = (
    ConstrainedPrompt('클라우드 컴퓨팅의 장점을 설명해주세요')
    .add_length('5개의 불릿포인트')
    .add_content('비용, 확장성, 보안 관점을 반드시 포함')
    .add_format('각 포인트는 한줄로 이모지로 시작')
    .add_style('IT 비전공 경영진을 대상, 전문 용어에 괄호로 설명 추가')
    .execute(temperature = 0.3)
)

In [17]:
print(result)

- 💰 **비용 효율성**: 클라우드 서비스는 초기 투자 비용을 줄이고, 사용한 만큼만 지불하는 구조로 운영비 절감에 기여합니다.  
- 📈 **확장성**: 필요에 따라 자원을 쉽게 늘리거나 줄일 수 있어, 비즈니스의 성장에 맞춰 유연하게 대응할 수 있습니다.  
- 🔒 **보안 강화를 위한 기술**: 클라우드는 전문 보안 시스템과 업데이트를 제공하여 데이터를 안전하게 보호합니다.  
- 🌍 **접근성 향상**: 인터넷이 연결된 어디서나 접근 가능해, 원격 근무나 글로벌 팀 간의 협업이 용이합니다.  
- ⚙️ **신속한 배포**: 새로운 애플리케이션이나 서비스 배포가 빠르게 이루어져, 시장 변화에 즉각적으로 대응할 수 있습니다.  


In [18]:
# 컨텍스트 제공
company_policy = """
[모두컴퍼니 재택근무 정책 v2.3]
- 주 3일 재택, 2일 출근 (화/목 필수 출근)
- 재택근무 시 오전 9시까지 Slack 상태 '업무중' 설정 필수
- 해외 원격근무는 최대 연속 2주까지 가능 (사전 승인 필요)
- 야간근무(22시 이후) 시 익일 오후 출근 가능
- 재택근무 장비 지원금: 연 100만원 (영수증 제출)
"""

In [19]:
question = '해외에서 한 달 동안 원격 근무할 수 있나요?'

In [20]:
with_context = f"""아래 회사 정책 문서를 참고하여 질문에 답하세요.
문서에 없는 내용은 "해당 정책 문서에 명시되어 있지 않습니다"라고 답하세요.

정책 문서:
\"\"\"{company_policy}\"\"\"

질문 : {question}"""

print(llm.invoke([HumanMessage(content=with_context)]).content)

해당 정책 문서에 명시되어 있지 않습니다.


In [21]:
def answer_with_context(context, question):
    clause = """
    중요 : 반드시 제공된 문서 내용만을 근거로 답변하세요.
    문서에 없는 내용은 "제공된 문서에 해당 정보가 없습니다"라고 답하세요.
    추측하거나 외부 지식을 사용하지 마세요."""
    
    prompt = f"""아래 참고 문서를 기반으로 질문에 답하세요.
    {clause}
    
    참고 문서:
    \"\"\"{company_policy}\"\"\"
    
    질문 : {question}
    
    답변 형식:
    - 답변 : [핵심 답변]
    - 근거 : [문서에서 관련 부분 인용]"""
    
    return llm.invoke([HumanMessage(content=prompt)]).content

In [22]:
answer_with_context(company_policy, question)

'- 답변 : 아니요, 한 달 동안 해외에서 원격 근무할 수 없습니다.\n- 근거 : "해외 원격근무는 최대 연속 2주까지 가능 (사전 승인 필요)"'

In [23]:
answer_with_context(company_policy, '식대 지원금은 얼마인가요?')

'- 답변 : 제공된 문서에 해당 정보가 없습니다.\n- 근거 : [문서에 식대 지원금에 대한 정보가 없습니다.]'

In [ ]:
# 자체 평가 시스템 프롬프트 : Automatic Prompt Engineer

In [ ]:
# Zero-shot, Few-shot

In [25]:
"""
식대 지원금에 대한 내용이 포함되어 있지 않습니다. -> 감정 분석해주세요 (Zero-shot)
Zero-shot : 예시 없이 바로 Ai에 보내는

식대 지원금에 대한 내용이 포함되어 있지 않습니다. : 긍정 -> 제공된 문서에 해당 정보가 없습니다. 감점 분석해주세요 (Few-shot)
"""

'\n식대 지원금에 대한 내용이 포함되어 있지 않습니다. -> 감정 분석해주세요 (Zero-shot)\nZero-shot : 예시 없이 바로 Ai에 보내는\n\n식대 지원금에 대한 내용이 포함되어 있지 않습니다. : 긍정 -> 제공된 문서에 해당 정보가 없습니다. 감점 분석해주세요 (Few-shot)\n'

In [26]:
categories = ["기술", "경제", "스포츠", "문화", "정치"]

news_articles = [
    "삼성전자가 차세대 AI 반도체 개발에 3조원을 투자한다고 발표했다.",
    "한국은행이 기준금리를 0.25%p 인하하며 경기 부양에 나섰다.",
    "손흥민이 프리미어리그 시즌 최다 도움을 기록하며 팀 승리를 이끌었다.",
    "국립현대미술관에서 한국 현대미술 50년 특별전이 개막했다."
]

In [27]:
# Zero-shot 예시 없이 분류 진행
classify_prompt = f"""당신은 뉴스 분류 전문가 입니다.
주어진 뉴스 기사를 다음 카테고리 중 하나로 분류하세요: {', '.join(categories)}
반드시 카테고리 이름만 출력하세요
"""

for article in news_articles:
    result = llm.invoke([
        SystemMessage(content=classify_prompt),
        HumanMessage(content=article)
    ]).content

    print(f"기사 : {article[:30]}")
    print(f"분류 : {result}\n")

기사 : 삼성전자가 차세대 AI 반도체 개발에 3조원을 투자한다
분류 : 기술

기사 : 한국은행이 기준금리를 0.25%p 인하하며 경기 부양에
분류 : 경제

기사 : 손흥민이 프리미어리그 시즌 최다 도움을 기록하며 팀 승
분류 : 스포츠

기사 : 국립현대미술관에서 한국 현대미술 50년 특별전이 개막했
분류 : 문화



In [29]:
# Few-shot 여러 예시를 가지고 진행
# 예시에 따른 토큰 비용이 나감
few_shot_messages = [
    SystemMessage(content='주어진 리뷰의 감정을 분석하세요'),
    # 예시
    HumanMessage(content='리뷰 : 이 제품 정말 최고입니다! 강력 추천해요'),
    AIMessage(content="{'sentiment' : '긍정', 'score' : 0.95, 'keywords' : ['최고', '강력 추천']}"),
    # 예시
    HumanMessage(content='리뷰 : 배송도 느리고 제품 품질도 형편없네요'),
    AIMessage(content="{'sentiment' : '부정', 'score' : 0.15, 'keywords' : ['느리고', '형편 없음']}"),
    # 예시
    HumanMessage(content='리뷰 : 가격 대비 괜찮지만, 기대했던 것보다는 아쉬워요'),
    AIMessage(content="{'sentiment' : '혼합', 'score' : 0.50, 'keywords' : ['괜찮지만', '아쉬워요']}"),

    # 실제 쿼리
    HumanMessage(content='리뷰 : 디자인은 예쁜데, 배터리가 빨리 닳아요. 전체적으로 보통입니다'),
]

print(llm.invoke(few_shot_messages).content)

{'sentiment' : '중립', 'score' : 0.50, 'keywords' : ['예쁜', '배터리 빨리 닳아', '보통']}


In [ ]:
examples = [
    {"informal": "내일 미팅 좀 미룰 수 있을까? 갑자기 일이 생겼어.",
     "formal": "안녕하세요. 내일 예정된 미팅 일정 변경을 요청드립니다. 긴급한 업무가 발생하여 조율이 필요합니다. 가능한 대체 일정을 알려주시면 감사하겠습니다."},
    {"informal": "그 보고서 다 했어? 빨리 보내줘.",
     "formal": "안녕하세요. 요청드렸던 보고서 진행 상황을 확인드립니다. 완료되셨다면 전달 부탁드리며, 추가 시간이 필요하시면 말씀해 주세요."},
    {"informal": "이번 프로젝트 결과 별로인데 어떻게 할까?",
     "formal": "안녕하세요. 이번 프로젝트 결과에 대해 논의가 필요합니다. 개선 방안을 함께 검토하기 위해 미팅을 잡는 것이 어떨까요?"}
]
test_messages = [
    "다음 주 워크숍 참석 못 할 것 같아. 다른 사람 보내도 돼?",
    "예산 좀 더 받을 수 있을까? 지금 부족해."
]